In [24]:
import pandas as pd
import os
raw_data_path = os.path.join("..", "data", "raw", "US-Exec_SOTU_2025.csv")
df_us = pd.read_csv(raw_data_path, encoding="latin1")
print("Columns in dataset:", df_us.columns)
print(df_us.head())

Columns in dataset: Index(['id', 'doc_count', 'filter_PolicySentence', 'date', 'oral_delivery',
       'outgoing', 'congress', 'president', 'pres_party', 'divided',
       'control_house', 'control_senate', 'year', 'month', 'day', 'source',
       'description', 'pap_majortopic', 'pap_subtopic', 'majortopic',
       'subtopic'],
      dtype='object')
    id  doc_count  filter_PolicySentence     date  oral_delivery  outgoing  \
0  1.0        1.0                    1.0  1/21/46            0.0       0.0   
1  2.0        2.0                    1.0  1/21/46            0.0       0.0   
2  3.0        3.0                    1.0  1/21/46            0.0       0.0   
3  4.0        4.0                    1.0  1/21/46            0.0       0.0   
4  5.0        5.0                    1.0  1/21/46            0.0       0.0   

   congress        president  pres_party  divided  ...  control_senate  \
0      79.0  Harry S. Truman       100.0      0.0  ...           100.0   
1      79.0  Harry S. Truman  

In [23]:
column_mapping = {
    "president": "speaker",
    "year": "year",
    "date": "date",
    "description": "text"
}
# Rename the columns
df_us_cleaned = df_us.rename(columns=column_mapping)
standard_cols = ['year', 'date', 'speaker', 'text']
df_us_cleaned = df_us_cleaned[[col for col in standard_cols if col in df_us_cleaned.columns]]
# clean up the text
df_us_cleaned['text'] = df_us_cleaned['text'].astype(str).str.strip()
df_us_cleaned = df_us_cleaned.dropna(subset=['text'])
output_path = os.path.join("..", "data", "clean", "us_speeches_clean.csv")
df_us_cleaned.to_csv(output_path, index=False, encoding="utf-8")
print(f"Successfully cleaned and saved {len(df_us_cleaned)} speeches to {output_path}!")
df_us_cleaned = df_us_cleaned.dropna(subset=['text'])
output_path = os.path.join("..", "data", "clean", "us_speeches_clean.csv")
df_us_cleaned.to_csv(output_path, index=False, encoding="utf-8")
print(f"Successfully cleaned and saved {len(df_us_cleaned)} speeches to {output_path}!")

Successfully cleaned and saved 26118 speeches to ..\data\clean\us_speeches_clean.csv!
Successfully cleaned and saved 26118 speeches to ..\data\clean\us_speeches_clean.csv!


In [4]:
import re
import os
import pandas as pd

us_speeches_path = os.path.join("..", "data", "clean", "us_speeches_clean.csv")
df_us_speeches = pd.read_csv(us_speeches_path, encoding="utf-8")

import matplotlib.pyplot as plt

us_speeches_path = os.path.join("..", "data", "clean", "us_speeches_clean.csv")
df_us_speeches = pd.read_csv(us_speeches_path, encoding="utf-8")

df_us_speeches = df_us_speeches.dropna(subset=['year']).copy()
df_us_speeches['year'] = df_us_speeches['year'].astype(int)
df_us_speeches['text'] = df_us_speeches['text'].astype(str)

YEAR_START, YEAR_END = 2004, 2021
df_us_filtered = df_us_speeches[(df_us_speeches['year'] >= YEAR_START) & (df_us_speeches['year'] <= YEAR_END)]

print(f"Analyzing US speeches from year {YEAR_START} to {YEAR_END}")



Analyzing US speeches from year 2004 to 2021


In [5]:
category_patterns = {
    category: r"\b(" + "|".join([re.escape(kw) for kw in keywords]) + r")\b"
    for category, keywords in CATEGORY_KEYWORDS.items()
}

total_per_year = df_us_filtered.groupby('year').size()

def get_us_category_counts(pattern, df):
    # Find matching rows using the exact same regex parameters as Cyrus
    matches = df[df["text"].str.contains(pattern, case=False, na=False, regex=True)]
    return matches.groupby('year').size()

category_year_counts = {
    category: get_us_category_counts(pattern, df_us_filtered)
    for category, pattern in category_patterns.items()
}

C:\Users\Mariam\AppData\Local\Temp\ipykernel_13888\2414806870.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = df[df["text"].str.contains(pattern, case=False, na=False, regex=True)]
C:\Users\Mariam\AppData\Local\Temp\ipykernel_13888\2414806870.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = df[df["text"].str.contains(pattern, case=False, na=False, regex=True)]
C:\Users\Mariam\AppData\Local\Temp\ipykernel_13888\2414806870.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  matches = df[df["text"].str.contains(pattern, case=False, na=False, regex=True)]
C:\Users\Mariam\AppData\Local\Temp\ipykernel_13888\2414806870.py:10: UserWarning: This pattern is interpreted as a regular expression, and has ma

In [ ]:
rows = []
for category, counts in category_year_counts.items():
    # Fill missing years with 0 to avoid NaNs in your trends
    counts = counts.reindex(range(YEAR_START, YEAR_END + 1), fill_value=0)
    totals = total_per_year.reindex(range(YEAR_START, YEAR_END + 1), fill_value=0)
    
    for year in range(YEAR_START, YEAR_END + 1):
        kw_speech_count = counts.loc[year]
        total_speech_count = totals.loc[year]
        hare = kw_speech_count / total_speech_count if total_speech_count > 0 else 0
        
        rows.append({
            "category": category,
            "year": year,
            "keyword_speeches": kw_speech_count,
            "total_speeches": total_speech_count,
            "normalised_share": hare
        })



NameError: name 'share' is not defined